# 🏥 MedAI — Multi-Disease Medical Imaging Classifier
### EfficientNet-B3 | 5 Separate Models | Hackathon Edition
**Diseases covered:** Chest X-Ray (NIH) · Brain Tumor (MRI) · COVID-19 (Radiography) · Malaria (Cell) · Bone Fractures (X-Ray)

---
> ⚡ Just run all cells top to bottom. Everything is automated.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q kaggle timm torchmetrics torchvision tqdm matplotlib seaborn scikit-learn Pillow

## 🔑 Step 2: Kaggle API Setup

In [ ]:
import os, json, shutil, zipfile, glob, time, copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ─── Kaggle credentials ───────────────────────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = "YOUR_USERNAME"
os.environ['KAGGLE_KEY']      = "YOUR_KEY"

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
cred_file  = kaggle_dir / 'kaggle.json'
cred_file.write_text(json.dumps({'username': os.environ['KAGGLE_USERNAME'],
                                  'key':      os.environ['KAGGLE_KEY']}))
cred_file.chmod(0o600)
print('✅ Kaggle credentials saved!')

## 📥 Step 3: Download All 5 Datasets from Kaggle

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

BASE_DIR = Path('/content/medai_data')
BASE_DIR.mkdir(exist_ok=True)

DATASETS = {
    'chest_xray':   ('nih-chest-xrays/sample',                        'chest_xray'),
    'brain_tumor':  ('masoudnickparvar/brain-tumor-mri-dataset',       'brain_tumor'),
    'covid19':      ('tawsifurrahman/covid19-radiography-database',    'covid19'),
    'malaria':      ('iarunava/cell-images-for-detecting-malaria',     'malaria'),
    'fracture':     ('bmadushanirodrigo/fracture-multi-region-x-ray-data', 'fracture'),
}

for key, (slug, folder) in DATASETS.items():
    dest = BASE_DIR / folder
    if dest.exists() and any(dest.iterdir()):
        print(f'⏭️  {key} already downloaded, skipping.')
        continue
    dest.mkdir(exist_ok=True)
    print(f'⬇️  Downloading {key} ...')
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    print(f'✅ {key} ready!')

print('\n🎉 All datasets downloaded!')

## 🗂️ Step 4: Dataset Path Discovery (Auto-detect folder structure)

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from PIL import Image
from sklearn.model_selection import train_test_split
from collections import Counter

def find_image_root(base_path, min_classes=2):
    """Recursively find the folder that contains class subfolders with images."""
    base = Path(base_path)
    IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    
    def has_images(p):
        return any(f.suffix.lower() in IMG_EXT for f in p.rglob('*') if f.is_file())
    
    def class_count(p):
        return sum(1 for d in p.iterdir() if d.is_dir() and has_images(d))
    
    # BFS to find best root
    candidates = [base]
    best = None
    for p in sorted(base.rglob('*')):
        if p.is_dir():
            n = class_count(p)
            if n >= min_classes:
                if best is None:
                    best = p
    return best or base

DATA_ROOTS = {}
for key, (_, folder) in DATASETS.items():
    root = find_image_root(BASE_DIR / folder)
    DATA_ROOTS[key] = root
    # Count classes
    classes = [d.name for d in root.iterdir() if d.is_dir()]
    print(f'📁 {key:15s} → {root.relative_to(BASE_DIR)}  |  classes: {classes}')

## 🔧 Step 5: Transforms & DataLoaders

In [ ]:
IMG_SIZE   = 300   # EfficientNet-B3 native size
BATCH_SIZE = 32
NUM_WORKERS= 2

# ─── Augmentation pipeline (aggressive for medical imgs) ─────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

def make_loaders(root, train_tf, val_tf, val_split=0.15, test_split=0.10, seed=42):
    """Create balanced train/val/test loaders from ImageFolder."""
    full_ds = ImageFolder(root=str(root))
    targets  = full_ds.targets
    indices  = list(range(len(full_ds)))
    
    # Stratified splits
    idx_train, idx_tmp = train_test_split(indices, test_size=val_split+test_split,
                                           stratify=targets, random_state=seed)
    t2 = [targets[i] for i in idx_tmp]
    ratio = test_split / (val_split + test_split)
    idx_val, idx_test = train_test_split(idx_tmp, test_size=ratio,
                                          stratify=t2, random_state=seed)
    
    class SplitDataset(Dataset):
        def __init__(self, base, idxs, tf):
            self.base, self.idxs, self.tf = base, idxs, tf
        def __len__(self): return len(self.idxs)
        def __getitem__(self, i):
            path, label = self.base.samples[self.idxs[i]]
            img = Image.open(path).convert('RGB')
            return self.tf(img), label
    
    train_ds = SplitDataset(full_ds, idx_train, train_tf)
    val_ds   = SplitDataset(full_ds, idx_val,   val_tf)
    test_ds  = SplitDataset(full_ds, idx_test,  val_tf)
    
    # Weighted sampler for class imbalance
    train_labels = [targets[i] for i in idx_train]
    counts  = Counter(train_labels)
    weights = [1.0 / counts[l] for l in train_labels]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    
    return train_loader, val_loader, test_loader, full_ds.classes

LOADERS = {}
CLASSES  = {}
for key, root in DATA_ROOTS.items():
    try:
        tl, vl, tel, cls = make_loaders(root, train_transform, val_transform)
        LOADERS[key] = (tl, vl, tel)
        CLASSES[key]  = cls
        print(f'✅ {key:15s} | {len(cls)} classes: {cls} | train={len(tl.dataset)} val={len(vl.dataset)} test={len(tel.dataset)}')
    except Exception as e:
        print(f'❌ {key}: {e}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n🖥️  Device: {DEVICE}')

## 🧠 Step 6: EfficientNet-B3 Model Builder

In [ ]:
import timm
import torch.nn as nn

def build_model(num_classes, pretrained=True, dropout=0.4):
    """EfficientNet-B3 with custom classification head."""
    model = timm.create_model(
        'efficientnet_b3',
        pretrained=pretrained,
        num_classes=0,          # remove default head
        drop_rate=dropout,
        drop_path_rate=0.2,
    )
    in_features = model.num_features  # 1536 for B3
    
    model.classifier = nn.Sequential(
        nn.BatchNorm1d(in_features),
        nn.Dropout(dropout),
        nn.Linear(in_features, 512),
        nn.SiLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(dropout / 2),
        nn.Linear(512, num_classes)
    )
    return model

# Freeze backbone initially, only train head (Phase 1)
def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

print('✅ Model builder ready!')
# Quick test
_m = build_model(4).to(DEVICE)
_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
print(f'   Test forward pass: input {list(_x.shape)} → output {list(_m(_x).shape)}')
del _m, _x

## 🏋️ Step 7: Training Engine (with LR scheduling, early stopping, GradScaler)

In [ ]:
from torch.cuda.amp import GradScaler, autocast
from torchmetrics import Accuracy, F1Score, AUROC

WEIGHTS_DIR = Path('/content/model_weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())
    return (total_loss / total, correct / total,
            torch.cat(all_preds), torch.cat(all_labels), torch.cat(all_probs))


def train_model(key, num_classes,
                phase1_epochs=5, phase2_epochs=20,
                lr_head=1e-3, lr_full=1e-4,
                patience=6):
    """Two-phase training: head warmup → full fine-tune with cosine annealing."""
    print(f'\n{"="*60}')
    print(f'  🏋️  Training: {key.upper()} | {num_classes} classes')
    print(f'{"="*60}')
    
    train_loader, val_loader, test_loader = LOADERS[key]
    model     = build_model(num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler()
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    save_path = WEIGHTS_DIR / f'{key}_efficientnet_b3.pth'
    
    best_val_acc = 0.
    best_state   = None

    # ── Phase 1: Train head only ──────────────────────────────────────────────
    print(f'\n  📌 Phase 1: Head warmup ({phase1_epochs} epochs)')
    freeze_backbone(model)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr_head, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr_head,
        steps_per_epoch=len(train_loader), epochs=phase1_epochs)
    
    for epoch in range(1, phase1_epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
        vl, va, *_ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        print(f'  Epoch {epoch:02d}/{phase1_epochs} | '
              f'train_loss={tl:.4f} acc={ta:.4f} | '
              f'val_loss={vl:.4f} acc={va:.4f}')
        if va > best_val_acc:
            best_val_acc = va
            best_state   = copy.deepcopy(model.state_dict())
    
    # ── Phase 2: Full fine-tune ───────────────────────────────────────────────
    print(f'\n  🚀 Phase 2: Full fine-tune ({phase2_epochs} epochs, early stopping patience={patience})')
    unfreeze_all(model)
    optimizer = torch.optim.AdamW([
        {'params': [p for n,p in model.named_parameters() if 'classifier' not in n],
         'lr': lr_full},
        {'params': model.classifier.parameters(),
         'lr': lr_full * 10}
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6)
    
    no_improve = 0
    for epoch in range(1, phase2_epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
        vl, va, *_ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        tag = ''
        if va > best_val_acc:
            best_val_acc = va
            best_state   = copy.deepcopy(model.state_dict())
            no_improve   = 0
            tag = ' ⭐ (best)'
        else:
            no_improve += 1
        print(f'  Epoch {epoch:02d}/{phase2_epochs} | '
              f'train_loss={tl:.4f} acc={ta:.4f} | '
              f'val_loss={vl:.4f} acc={va:.4f}{tag}')
        if no_improve >= patience:
            print(f'  ⏹️  Early stopping at epoch {epoch}')
            break
    
    # ── Save best weights ─────────────────────────────────────────────────────
    model.load_state_dict(best_state)
    torch.save({
        'model_state_dict': best_state,
        'classes':          CLASSES[key],
        'num_classes':      num_classes,
        'architecture':     'efficientnet_b3',
        'img_size':         IMG_SIZE,
        'best_val_acc':     best_val_acc,
        'dataset_key':      key,
    }, save_path)
    print(f'\n  💾 Saved → {save_path}')
    print(f'  🏆 Best Val Accuracy: {best_val_acc:.4f}')
    
    return model, history, test_loader

print('✅ Training engine ready!')

## 📊 Step 8: Evaluation & Metrics Visualizer

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def plot_results(key, model, history, test_loader, classes):
    criterion = nn.CrossEntropyLoss()
    _, test_acc, preds, labels, probs = evaluate(model, test_loader, criterion, DEVICE)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle(f'📊 {key.upper()} Results | Test Acc: {test_acc:.4f}', fontsize=14, fontweight='bold')
    
    # Loss curve
    ax = axes[0]
    ax.plot(history['train_loss'], label='Train Loss', color='royalblue')
    ax.plot(history['val_loss'],   label='Val Loss',   color='tomato')
    ax.set_title('Loss Curve'); ax.legend(); ax.set_xlabel('Epoch')
    
    # Accuracy curve
    ax = axes[1]
    ax.plot(history['train_acc'], label='Train Acc', color='royalblue')
    ax.plot(history['val_acc'],   label='Val Acc',   color='tomato')
    ax.set_title('Accuracy Curve'); ax.legend(); ax.set_xlabel('Epoch')
    
    # Confusion matrix
    ax = axes[2]
    cm = confusion_matrix(labels.numpy(), preds.numpy())
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig(WEIGHTS_DIR / f'{key}_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\n📋 Classification Report — {key.upper()}')
    print(classification_report(labels.numpy(), preds.numpy(), target_names=classes))
    print(f'  ✅ Test Accuracy: {test_acc:.4f}')
    return test_acc

print('✅ Evaluation tools ready!')

## 🚀 Step 9: TRAIN ALL 5 MODELS
> This is the big one. Each model trains sequentially. Go grab a coffee ☕

In [ ]:
RESULTS = {}
TRAINED_MODELS = {}

for key in LOADERS.keys():
    num_classes = len(CLASSES[key])
    model, history, test_loader = train_model(
        key, num_classes,
        phase1_epochs=5,
        phase2_epochs=25,
        patience=7
    )
    test_acc = plot_results(key, model, history, test_loader, CLASSES[key])
    RESULTS[key] = test_acc
    TRAINED_MODELS[key] = model
    torch.cuda.empty_cache()  # free VRAM between models

print('\n' + '='*60)
print('  🏆 FINAL RESULTS SUMMARY')
print('='*60)
for k, acc in RESULTS.items():
    bar = '█' * int(acc * 20)
    print(f'  {k:20s} | {bar:<20s} | {acc:.4f} ({acc*100:.1f}%)')

## 💾 Step 10: Verify Saved Weights

In [ ]:
print('📦 Saved model weights:')
total_size = 0
for pth_file in sorted(WEIGHTS_DIR.glob('*.pth')):
    size_mb = pth_file.stat().st_size / (1024**2)
    total_size += size_mb
    # Load and verify
    ckpt = torch.load(pth_file, map_location='cpu')
    print(f'  ✅ {pth_file.name:45s} | {size_mb:.1f} MB | '
          f'classes={ckpt["num_classes"]} | best_val_acc={ckpt["best_val_acc"]:.4f}')
print(f'\n  Total size: {total_size:.1f} MB')

## 🔮 Step 11: Inference Function (For Frontend Integration)

In [ ]:
def load_model_for_inference(pth_path, device='cpu'):
    """Load a saved .pth model for inference."""
    ckpt  = torch.load(pth_path, map_location=device)
    model = build_model(ckpt['num_classes'])
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    model.to(device)
    return model, ckpt['classes']

infer_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict(model, image_path_or_pil, classes, device='cpu', top_k=3):
    """
    Run inference on a single image.
    Returns: dict with top_k predictions and confidence scores.
    """
    if isinstance(image_path_or_pil, (str, Path)):
        img = Image.open(image_path_or_pil).convert('RGB')
    else:
        img = image_path_or_pil.convert('RGB')
    
    tensor = infer_transform(img).unsqueeze(0).to(device)
    logits = model(tensor)
    probs  = torch.softmax(logits, dim=1)[0]
    
    top_probs, top_idxs = probs.topk(min(top_k, len(classes)))
    results = [
        {'class': classes[i], 'confidence': round(p.item() * 100, 2)}
        for i, p in zip(top_idxs.tolist(), top_probs.tolist())
    ]
    return {
        'prediction':   results[0]['class'],
        'confidence':   results[0]['confidence'],
        'top_k':        results,
    }

# ── Quick demo inference ──────────────────────────────────────────────────────
print('🔮 Inference function ready!')
print('\nUsage example:')
print("""
  model, classes = load_model_for_inference('/content/model_weights/malaria_efficientnet_b3.pth')
  result = predict(model, 'path/to/cell_image.png', classes)
  print(result)
  # → {'prediction': 'Parasitized', 'confidence': 97.3, 'top_k': [...]}
""")

## ☁️ Step 12: Save Weights to Google Drive (Optional)

In [ ]:
# Run this cell if you want to back up weights to Google Drive
SAVE_TO_DRIVE = True  # Set False to skip

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_dest = Path('/content/drive/MyDrive/MedAI_Weights')
    drive_dest.mkdir(exist_ok=True)
    
    for pth_file in WEIGHTS_DIR.glob('*.pth'):
        dest = drive_dest / pth_file.name
        shutil.copy(pth_file, dest)
        print(f'  ✅ Copied {pth_file.name} → Google Drive')
    
    # Also copy result plots
    for png_file in WEIGHTS_DIR.glob('*.png'):
        shutil.copy(png_file, drive_dest / png_file.name)
    
    print(f'\n🎉 All weights saved to Google Drive: MyDrive/MedAI_Weights/')
else:
    print('⏭️  Skipping Drive save.')

## ✅ Done!

| Model | Weights File |
|---|---|
| Chest X-Ray (NIH) | `chest_xray_efficientnet_b3.pth` |
| Brain Tumor (MRI) | `brain_tumor_efficientnet_b3.pth` |
| COVID-19 | `covid19_efficientnet_b3.pth` |
| Malaria | `malaria_efficientnet_b3.pth` |
| Fracture | `fracture_efficientnet_b3.pth` |

All weights saved to `/content/model_weights/` and optionally Google Drive.

### Frontend integration
Just call `load_model_for_inference(path)` then `predict(model, image, classes)` — returns JSON ready to pipe into your UI.

---
**Good luck at the hackathon! 🏆**